# ML-02 â€” Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/broskell/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring**

The starter dataset — `content_refresh_anonymized.csv` — was built exactly for this lane. It
ships 30,000 content items with 90-day search and engagement signals, freshness tiers, trend
direction, and a declining-label proxy. The starter pipeline already proves a machine-learned
ranking beats a fixed-rule baseline on this slice (Precision@50: 0.74 vs 0.24). That gives me
a running start: I can improve the label, add stronger validation, and ship a ranked review
queue a content team could actually use.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
trend_counts = df['trend_direction'].value_counts()
print('=== Lane 2 evidence ===')
print(f'Total rows: {len(df):,}')
print(f'Distinct clients: {df["client_id"].nunique()}')
for val, cnt in trend_counts.items():
    print(f'  {val:8s}  {cnt:>6,}  ({cnt/len(df)*100:.1f}%)')
print(f'\nDeclining (positive class): {trend_counts.get("down",0):,}')
stale = df[df['days_since_last_update'] >= 180]
print(f'Stale pages (>=180 days since update): {len(stale):,}  ({len(stale)/len(df)*100:.1f}%)')
print(f'  Mean impressions: {stale["impressions_90d"].mean():,.0f}')


=== Lane 2 evidence ===
Total rows: 30,000
Distinct clients: 32
  down      16,262  (54.2%)
  stable     5,962  (19.9%)
  up         4,388  (14.6%)
  new        2,236  (7.5%)
  flat       1,152  (3.8%)

Declining (positive class): 16,262
Stale pages (>=180 days since update): 174  (0.6%)
  Mean impressions: 1,172


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which content pages should a human editor review first for refresh, expansion,
or protection — given limited editorial capacity?

**Who acts:** A content or SEO editor who can review roughly 50 pages per cycle. They need a
ranked list, not a flood.

**What does a wrong recommendation cost?** Two kinds of error: a **false positive** wastes an
editor's limited time on a page that did not need attention — the editor could have spent that
hour on a genuinely at-risk page. A **false negative** misses a page that is quietly losing
traffic; if left unchecked, that page continues to decay, losing clicks and sessions week after
week. Because editorial time is scarcer than missed opportunities in this dataset (over half the
inventory shows a declining trend), I will prioritize high-precision top-K recommendations.

In [4]:
declining = df[df["trend_direction"] == "down"]
with_impressions = declining[declining["impressions_90d"] >= 100]
print(f"Declining pages with >= 100 impressions: {len(with_impressions):,}")
print(f"Fraction of all pages: {len(with_impressions)/len(df)*100:.1f}%")
print(f"A reviewer handling 50 pages per cycle needs a PRECISE top-50 ranking.")
print(f"Median impressions among declining pages: {declining['impressions_90d'].median():,.0f}")


Declining pages with >= 100 impressions: 13,152
Fraction of all pages: 43.8%
A reviewer handling 50 pages per cycle needs a PRECISE top-50 ranking.
Median impressions among declining pages: 961


## 3. Quick look at the data (2-3 real numbers)

The starter dataset has 30,000 rows across 32 pseudonymized clients. Every row carries trailing
90-day signals — impressions, clicks, sessions, engagement, freshness, and a trend label. The
numbers below are computed live from `data/raw/content_refresh_anonymized.csv`.

In [6]:
print("=== Quick statistics (computed from data) ===")
print(f"Rows:                               {len(df):,}")
print(f"Distinct clients:                   {df['client_id'].nunique()}")
print(f"Distinct content types:             {df['content_type'].nunique()}")
print()

# Class balance
pos = (df["trend_direction"] == "down").sum()
neg = (df["trend_direction"] != "down").sum()
print(f"Declining (positive class):         {pos:,}  ({pos/len(df)*100:.1f}%)")
print(f"Not declining (negative class):     {neg:,}  ({neg/len(df)*100:.1f}%)")
print()

# Engagement and freshness
has_sessions = df[df['sessions_90d'] > 0]
print(f"Pages with ANY sessions (90d):       {len(has_sessions):,}  ({len(has_sessions)/len(df)*100:.1f}%)")
print(f"  Median sessions:                   {has_sessions['sessions_90d'].median():.0f}")
print(f"  Median sessions (declining):       {has_sessions[has_sessions['trend_direction']=='down']['sessions_90d'].median():.0f}")
print()

# Freshness risk
stale180 = df[df['days_since_last_update'] >= 180]
stale365 = df[df['days_since_last_update'] >= 365]
print(f"Stale >= 180 days since update:      {len(stale180):,}  ({len(stale180)/len(df)*100:.1f}%)")
print(f"Stale >= 365 days since update:      {len(stale365):,}  ({len(stale365)/len(df)*100:.1f}%)")
print(f"  Mean impressions (stale 180+):     {stale180['impressions_90d'].mean():,.0f}")
print()

# CTR context
ctr_mask = (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['impressions_90d'] >= 500)
ctr_candidates = df[ctr_mask]
low_ctr = ctr_candidates[ctr_candidates['ctr'] < 0.5]
print(f"Visible + low-CTR review candidates:  {len(low_ctr):,}")
print()

# Summary
print(f"Mean content age (days):            {df['content_age_days'].mean():.0f}")
print(f"Median content age (days):           {df['content_age_days'].median():.0f}")
print(f"Mean impressions (90d):             {df['impressions_90d'].mean():,.0f}")
print(f"Median impressions (90d):            {df['impressions_90d'].median():,.0f}")


=== Quick statistics (computed from data) ===
Rows:                               30,000
Distinct clients:                   32
Distinct content types:             3

Declining (positive class):         16,262  (54.2%)
Not declining (negative class):     13,738  (45.8%)

Pages with ANY sessions (90d):       30,000  (100.0%)
  Median sessions:                   7
  Median sessions (declining):       8

Stale >= 180 days since update:      174  (0.6%)
Stale >= 365 days since update:      5  (0.0%)
  Mean impressions (stale 180+):     1,172

Visible + low-CTR review candidates:  9,759

Mean content age (days):            256
Median content age (days):           236
Mean impressions (90d):             5,200
Median impressions (90d):            731


## 4. Careful words: what I can and can't claim

**What this work CAN claim (observed, directional, decision-support):**

- Which pages show declining search visibility or engagement, based on observed trailing-90-day
  signals.
- That a machine-learned ranking prioritizes pages more precisely than a hand-tuned rule
  baseline, measured by Precision@50 on held-out clients.
- That certain page characteristics (freshness, position, engagement rate, CTR) are associated
  with decline risk — association, not causation.
- A ranked review queue with reason codes a human editor can inspect and act on.

**What this work CANNOT claim (and never will):**

- That editing a page will cause it to recover — the data is observational. Proving causal
  recovery requires an experiment or a difference-in-differences design, which this release
  does not support alone.
- Anything about Google's algorithm, AI rankings, or "what search engines want."
- That the model "predicts the future" — our label is computed from the SAME 90-day window
  as the features, so this is a proxy-label ranking, not a true future-outcome prediction.
  The capstone can improve this with a proper feature-window vs. target-window split.

In [8]:
# Notebook summary — all numbers above were computed from the data, not hardcoded.
import datetime
print(f"Notebook execution completed: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Notebook execution completed: 2026-07-30 12:45:46


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled â€” markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime â†’ Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` â€” then submit your repo URL on the card. Done.